In [ ]:
%%writefile matrix_mul.cu

#include <stdio.h>
#include <cuda_runtime.h>

#define N 16

// CUDA Kernel for Matrix Multiplication
__global__ void matrixMulKernel(float *A, float *B, float *C, int n)
{
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < n && col < n)
    {
        float sum = 0.0f;

        for (int k = 0; k < n; ++k)
        {
            sum += A[row * n + k] * B[k * n + col];
        }

        C[row * n + col] = sum;
    }
}

// Main Function
int main()
{
    int size = N * N * sizeof(float);

    // Host matrices
    float *h_A = (float *)malloc(size);
    float *h_B = (float *)malloc(size);
    float *h_C = (float *)malloc(size);

    // Initialize matrices
    for (int i = 0; i < N * N; i++)
    {
        h_A[i] = 1.0f;
        h_B[i] = 2.0f;
    }

    // Device matrices
    float *d_A, *d_B, *d_C;

    cudaMalloc((void **)&d_A, size);
    cudaMalloc((void **)&d_B, size);
    cudaMalloc((void **)&d_C, size);

    // Copy data to GPU
    cudaMemcpy(d_A, h_A, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, size, cudaMemcpyHostToDevice);

    // Kernel configuration
    dim3 threadsPerBlock(16, 16);

    dim3 blocksPerGrid(
        (N + 15) / 16,
        (N + 15) / 16
    );

    // Launch kernel
    matrixMulKernel<<<blocksPerGrid, threadsPerBlock>>>(
        d_A,
        d_B,
        d_C,
        N
    );

    // Copy result back
    cudaMemcpy(
        h_C,
        d_C,
        size,
        cudaMemcpyDeviceToHost
    );

    // Print top-left 4x4 matrix
    printf("Top-left 4x4 Result Matrix:\n");

    for (int i = 0; i < 4; i++)
    {
        for (int j = 0; j < 4; j++)
        {
            printf("%.1f ", h_C[i * N + j]);
        }

        printf("\n");
    }

    // Free memory
    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    free(h_A);
    free(h_B);
    free(h_C);

    return 0;
}

Writing matrix_mul.cu


In [ ]:
!nvcc matrix_mul.cu -o matrix_mul

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [ ]:
!./matrix_mul

Top-left 4x4 Result Matrix:
32.0 32.0 32.0 32.0 
32.0 32.0 32.0 32.0 
32.0 32.0 32.0 32.0 
32.0 32.0 32.0 32.0 
